# Parameter sweep analysis — lambda (λ) for flame designs

This notebook analyzes the effect of the Ledidi regularization parameter
**λ** on flame design optimization outcomes.

In the AkitaSF pipeline, λ controls the weight of the input loss term
(L1 distance between the original and edited sequence) relative to the
output loss (contact map loss). Flame designs optimize a **single central
bin**, targeting a vertical stripe of elevated contact frequency in the
upper half of the contact map centred on the middle column.

The flame score (`flame3`) is the mean contact frequency in a 3-bin-wide
vertical stripe centred on the map's middle column in the upper half of
the map. A design is considered successful if `flame3_edited −
flame3_orig > 0`.

The sweep was run on folds 0–3 across λ values:
`[0.01, 0.1, 1.0, 10.0, 100.0, 140.0, 200.0]`.

## Outline
1. Load sweep results across λ values and folds
2. Fraction of windows with no edits accepted per λ
3. Fraction of windows with no flame score improvement per λ
4. Overall success rate per λ (edits accepted AND flame score improved)
5. Summary statistics: mean edits, flame score difference, and CTCF count per λ

In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("/home1/smaruj/akita_semifreddo/"))
from utils.df_utils import load_parameter_results

In [2]:
_PROJ     = "/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita"
RESULTS_DIR = f"{_PROJ}/optimizations/flames"

FOLDS   = [0, 1, 2, 3]
LAMBDAS = [0.01, 0.1, 1.0, 10.0, 100.0, 140.0, 200.0]

In [3]:
# Lambda sweep
df_lambda = load_parameter_results(RESULTS_DIR, "lambda", LAMBDAS, folds=FOLDS)

Total windows loaded: 1218


In [4]:
df_lambda["no_edits"] = df_lambda["n_edits"] == 0

no_edits_summary = (
    df_lambda.groupby("lambda")["no_edits"]
    .agg(n_total="count", n_no_edits="sum")
    .assign(pct_no_edits=lambda x: 100 * x["n_no_edits"] / x["n_total"])
)
print(no_edits_summary.to_string())

        n_total  n_no_edits  pct_no_edits
lambda                                   
0.01        174           0      0.000000
0.10        174           0      0.000000
1.00        174           0      0.000000
10.00       174           0      0.000000
100.00      174           7      4.022989
140.00      174          11      6.321839
200.00      174          18     10.344828


In [5]:
df_lambda["flame_diff"] = df_lambda["flame3_edited"] - df_lambda["flame3_orig"]
df_lambda["no_improvement"] = df_lambda["flame_diff"] <= 0

no_improvement_summary = (
    df_lambda.groupby("lambda")["no_improvement"]
    .agg(n_total="count", n_no_improvement="sum")
    .assign(pct_no_improvement=lambda x: 100 * x["n_no_improvement"] / x["n_total"])
)
print(no_improvement_summary.to_string())

        n_total  n_no_improvement  pct_no_improvement
lambda                                               
0.01        174                 0            0.000000
0.10        174                 0            0.000000
1.00        174                 0            0.000000
10.00       174                 0            0.000000
100.00      174                 7            4.022989
140.00      174                11            6.321839
200.00      174                18           10.344828


In [6]:
success_rate = (
    df_lambda.assign(success=~df_lambda["no_edits"] & ~df_lambda["no_improvement"])
    .groupby("lambda")["success"]
    .agg(n_total="count", n_success="sum")
    .assign(pct_success=lambda x: 100 * x["n_success"] / x["n_total"])
)
print(success_rate.to_string())

        n_total  n_success  pct_success
lambda                                 
0.01        174        174   100.000000
0.10        174        174   100.000000
1.00        174        174   100.000000
10.00       174        174   100.000000
100.00      174        167    95.977011
140.00      174        163    93.678161
200.00      174        156    89.655172


In [7]:
summary = (
    df_lambda.groupby("lambda")
    .agg(
        n_success        = ("n_edits",          "count"),
        mean_n_edits     = ("n_edits",          "mean"),
        mean_flame_score_diff  = ("flame_diff",        "mean"),
        mean_CTCFs       = ("CTCFs_num",         "mean"),
    )
    .round(3)
)
print(summary.to_string())

        n_success  mean_n_edits  mean_flame_score_diff  mean_CTCFs
lambda                                                            
0.01          174       744.730                  0.715       4.310
0.10          174       737.747                  0.725       4.678
1.00          174       672.063                  0.709       4.425
10.00         174       376.879                  0.704       4.460
100.00        174        80.115                  0.662       3.276
140.00        174        58.540                  0.646       2.937
200.00        174        41.810                  0.614       2.626
